# PhishGuard — ML Training
Run all cells top to bottom (Kernel → Restart & Run All). All numbers come from real execution; no outputs are pre-filled.

## 01 Import dataset
Edit `config.py` to change the dataset path / column names. Labels: 0 = legitimate, 1 = phishing.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..')); os.chdir('..')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import config
from src.preprocessing import load_dataset, build_matrix
df = load_dataset()
df.head()

## 02 Dataset cleaning
`load_dataset` drops missing rows and **duplicate URLs** (prevents the same URL landing in both train and test).

In [ ]:
print(df.shape); print(df['label'].value_counts()); df.isna().sum()

## 03 Exploratory data analysis

In [ ]:
sns.countplot(x='label', data=df); plt.title('Class balance (0=legit, 1=phishing)'); plt.show()
df['len']=df.url.str.len(); sns.boxplot(x='label', y='len', data=df); plt.show()

## 04 Feature extraction
The **same** `extract_features` used by the Flask app produces the training matrix, guaranteeing identical feature definitions and order.

In [ ]:
X, y = build_matrix(df)
print(X.shape); X.head()

## 05 Feature distribution

In [ ]:
X.assign(label=y).groupby('label')[['url_length','num_subdomains','keyword_count','is_https','suspicious_tld','is_ip_host','entropy']].mean().T

## 06 Train/test split & data-leakage prevention
* Stratified 80/20 split, fixed seed.
* URLs de-duplicated beforehand; the label is never a feature; the split happens **before** fitting; medians for explanations use training data only.
* All reported metrics use only the unseen test set.

In [ ]:
from src.model_utils import split_data
Xtr, Xte, ytr, yte = split_data(X, y)
print(len(Xtr), len(Xte), set(df.url[:0]))

## 07 Model training

In [ ]:
from src.model_utils import get_models, evaluate
models = get_models()
for n, m in models.items(): m.fit(Xtr, ytr)
list(models)

## 08 Model comparison
False negatives (phishing labelled legitimate) are the costliest error in security, so watch **recall/FNR**, not just accuracy.

In [ ]:
res = {n: evaluate(m, Xte, yte) for n, m in models.items()}
cmp = pd.DataFrame(res).T.drop(columns='confusion_matrix'); cmp.round(4)

## 09 Hyperparameter tuning (Random Forest)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
gs = GridSearchCV(RandomForestClassifier(random_state=42), {'n_estimators':[100,200],'max_depth':[None,10,20]}, cv=5, scoring='f1', n_jobs=-1)
gs.fit(Xtr, ytr); print(gs.best_params_, gs.best_score_)

## 10 Model evaluation (tuned RF, unseen test data)

In [ ]:
from sklearn.metrics import classification_report, roc_curve
best_rf = gs.best_estimator_; print(evaluate(best_rf, Xte, yte)); print(classification_report(yte, best_rf.predict(Xte)))
fpr,tpr,_ = roc_curve(yte, best_rf.predict_proba(Xte)[:,1]); plt.plot(fpr,tpr); plt.plot([0,1],[0,1],'--'); plt.title('ROC'); plt.show()

## 11 Feature importance (computed, not invented)

In [ ]:
imp = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values()
imp.tail(15).plot.barh(figsize=(7,5), title='Random Forest feature importance'); plt.show()

## 12 Confusion matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_estimator(best_rf, Xte, yte, display_labels=['Legit','Phish'], cmap='Blues'); plt.show()

## 13 Save model
Saves the bundle (model + feature list + training medians) and `metrics.json` to `models/`, which the Flask app loads.

In [ ]:
from src.model_utils import train_and_save
results, best, importance = train_and_save(); print('Saved best model:', best)

## 14 Test custom URLs

In [ ]:
from src.prediction import analyze_url
for u in ['https://www.wikipedia.org/','http://192.168.4.7/login/verify','http://secure-login-paypal-verify.xyz/account']:
    r = analyze_url(u); print(u, '->', r['verdict'], r['score'], r['prob'])